<a href="https://colab.research.google.com/github/Nifal123/financial-/blob/main/04_Backtesting_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook 4 - Portfolio Backtesting Engine

Author: Your Name
Date: May 2026

What This Notebook Covers
- Walk-forward backtesting with monthly rebalancing
- Compare all 4 strategies vs 60/40 benchmark
- Cumulative returns over 6 years
- Year by year performance breakdown
- Drawdown analysis per strategy
- Full performance tearsheet


In [1]:
!pip install yfinance plotly scipy -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/portfolio_optimizer/'

daily_returns  = pd.read_csv(f'{save_path}daily_returns.csv', index_col=0, parse_dates=True)
monthly_returns= pd.read_csv(f'{save_path}monthly_returns.csv', index_col=0, parse_dates=True)
prices         = pd.read_csv(f'{save_path}prices.csv', index_col=0, parse_dates=True)

RISK_FREE_RATE = 0.05
TRADING_DAYS   = 252
NUM_ASSETS     = len(daily_returns.columns)
ASSET_NAMES    = list(daily_returns.columns)

print("Data loaded successfully")
print("Assets    :", ASSET_NAMES)
print("Date range:", daily_returns.index[0].date(), "to", daily_returns.index[-1].date())

Mounted at /content/drive
Data loaded successfully
Assets    : ['US Equities', 'Intl Equities', 'Long Bonds', 'Agg Bonds', 'Gold', 'Commodities', 'Real Estate']
Date range: 2018-01-03 to 2024-12-30


In [2]:
# Core portfolio functions — same as Notebook 3

def portfolio_return(weights, mean_returns):
    return np.dot(weights, mean_returns)

def portfolio_volatility(weights, cov_matrix):
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

def portfolio_sharpe(weights, mean_returns, cov_matrix, rf=RISK_FREE_RATE):
    ret = portfolio_return(weights, mean_returns)
    vol = portfolio_volatility(weights, cov_matrix)
    return (ret - rf) / vol

def get_max_sharpe_weights(returns_data):
    n   = len(returns_data.columns)
    mu  = returns_data.mean() * TRADING_DAYS
    cov = returns_data.cov() * TRADING_DAYS

    def neg_sharpe(w):
        return -portfolio_sharpe(w, mu, cov)

    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds      = tuple((0, 1) for _ in range(n))
    x0          = np.array([1/n] * n)
    result      = minimize(neg_sharpe, x0, method='SLSQP',
                           bounds=bounds, constraints=constraints)
    return result.x if result.success else x0

def get_min_variance_weights(returns_data):
    n   = len(returns_data.columns)
    cov = returns_data.cov() * TRADING_DAYS

    def port_vol(w):
        return portfolio_volatility(w, cov)

    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds      = tuple((0, 1) for _ in range(n))
    x0          = np.array([1/n] * n)
    result      = minimize(port_vol, x0, method='SLSQP',
                           bounds=bounds, constraints=constraints)
    return result.x if result.success else x0

def get_risk_parity_weights(returns_data):
    n   = len(returns_data.columns)
    cov = returns_data.cov() * TRADING_DAYS

    def rp_error(w):
        w       = np.array(w)
        vol     = portfolio_volatility(w, cov)
        mrc     = np.dot(cov, w) / vol
        rc      = w * mrc
        target  = vol / n
        return np.sum((rc - target) ** 2)

    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds      = tuple((0.01, 1) for _ in range(n))
    x0          = np.array([1/n] * n)
    result      = minimize(rp_error, x0, method='SLSQP',
                           bounds=bounds, constraints=constraints,
                           options={'maxiter': 1000, 'ftol': 1e-12})
    w = result.x if result.success else x0
    return w / w.sum()

print("All portfolio functions defined successfully")

All portfolio functions defined successfully


In [3]:
# Walk-Forward Backtesting Engine
# Every month we look back 12 months, recalculate optimal weights
# then apply those weights to the NEXT month
# This is realistic — we never use future data

def run_backtest(daily_returns, strategy='max_sharpe', lookback_months=12):

    print(f"Running backtest: {strategy} | Lookback: {lookback_months} months")

    monthly    = daily_returns.resample('ME').apply(lambda x: (1+x).prod()-1)
    port_rets  = []
    port_dates = []
    all_weights= []

    for i in range(lookback_months, len(monthly)):

        # Training window — past 12 months only
        train = daily_returns[
            (daily_returns.index >= monthly.index[i - lookback_months]) &
            (daily_returns.index <  monthly.index[i])
        ]

        if len(train) < 20:
            continue

        # Calculate weights based on strategy
        if strategy == 'max_sharpe':
            weights = get_max_sharpe_weights(train)
        elif strategy == 'min_variance':
            weights = get_min_variance_weights(train)
        elif strategy == 'risk_parity':
            weights = get_risk_parity_weights(train)
        elif strategy == 'equal_weight':
            weights = np.array([1/NUM_ASSETS] * NUM_ASSETS)
        else:
            weights = np.array([1/NUM_ASSETS] * NUM_ASSETS)

        # Apply weights to that month's actual return
        month_ret = monthly.iloc[i].values
        port_ret  = np.dot(weights, month_ret)

        port_rets.append(port_ret)
        port_dates.append(monthly.index[i])
        all_weights.append(weights)

    results = pd.Series(port_rets, index=port_dates, name=strategy)
    return results, all_weights

# Run all 4 strategies
print("Starting backtests — this takes about 1-2 minutes...")
print("")

bt_max_sharpe,   wt_ms = run_backtest(daily_returns, 'max_sharpe')
bt_min_variance, wt_mv = run_backtest(daily_returns, 'min_variance')
bt_risk_parity,  wt_rp = run_backtest(daily_returns, 'risk_parity')
bt_equal_weight, wt_ew = run_backtest(daily_returns, 'equal_weight')

print("")
print("All backtests complete")

Starting backtests — this takes about 1-2 minutes...

Running backtest: max_sharpe | Lookback: 12 months
Running backtest: min_variance | Lookback: 12 months
Running backtest: risk_parity | Lookback: 12 months
Running backtest: equal_weight | Lookback: 12 months

All backtests complete


In [4]:
# Build the 60/40 benchmark
# 60% SPY (US Equities) + 40% TLT (Long Bonds)
# This is the standard benchmark every institutional PM is compared against

spy_returns = daily_returns['US Equities'].resample('ME').apply(lambda x: (1+x).prod()-1)
tlt_returns = daily_returns['Long Bonds'].resample('ME').apply(lambda x: (1+x).prod()-1)

benchmark_60_40 = (0.60 * spy_returns + 0.40 * tlt_returns)

# Align all series to same date range
common_start = max(
    bt_max_sharpe.index[0],
    bt_min_variance.index[0],
    bt_risk_parity.index[0],
    bt_equal_weight.index[0]
)

bt_max_sharpe   = bt_max_sharpe[bt_max_sharpe.index >= common_start]
bt_min_variance = bt_min_variance[bt_min_variance.index >= common_start]
bt_risk_parity  = bt_risk_parity[bt_risk_parity.index >= common_start]
bt_equal_weight = bt_equal_weight[bt_equal_weight.index >= common_start]
benchmark_60_40 = benchmark_60_40[benchmark_60_40.index >= common_start]
benchmark_60_40 = benchmark_60_40.reindex(bt_max_sharpe.index)

print("60/40 benchmark built successfully")
print(f"Backtest period : {common_start.date()} to {bt_max_sharpe.index[-1].date()}")
print(f"Monthly periods : {len(bt_max_sharpe)}")

60/40 benchmark built successfully
Backtest period : 2019-01-31 to 2024-12-31
Monthly periods : 72


In [5]:
# Performance metrics function for each backtest

def backtest_metrics(monthly_returns_series, rf=RISK_FREE_RATE, name='Strategy'):

    r           = monthly_returns_series.dropna()
    monthly_rf  = rf / 12

    ann_return  = (1 + r).prod() ** (12/len(r)) - 1
    ann_vol     = r.std() * np.sqrt(12)
    sharpe      = (ann_return - rf) / ann_vol if ann_vol != 0 else 0

    downside    = r[r < monthly_rf].std() * np.sqrt(12)
    sortino     = (ann_return - rf) / downside if downside != 0 else 0

    cumulative  = (1 + r).cumprod()
    rolling_max = cumulative.cummax()
    drawdown    = (cumulative - rolling_max) / rolling_max
    max_dd      = drawdown.min()
    calmar      = ann_return / abs(max_dd) if max_dd != 0 else 0

    total_ret   = (1 + r).prod() - 1
    win_rate    = (r > 0).sum() / len(r)

    return {
        'Strategy'        : name,
        'Ann. Return %'   : round(ann_return * 100, 2),
        'Total Return %'  : round(total_ret * 100, 2),
        'Volatility %'    : round(ann_vol * 100, 2),
        'Sharpe Ratio'    : round(sharpe, 3),
        'Sortino Ratio'   : round(sortino, 3),
        'Calmar Ratio'    : round(calmar, 3),
        'Max Drawdown %'  : round(max_dd * 100, 2),
        'Win Rate %'      : round(win_rate * 100, 1),
    }

results_table = pd.DataFrame([
    backtest_metrics(bt_max_sharpe,   name='Max Sharpe'),
    backtest_metrics(bt_min_variance, name='Min Variance'),
    backtest_metrics(bt_risk_parity,  name='Risk Parity'),
    backtest_metrics(bt_equal_weight, name='Equal Weight'),
    backtest_metrics(benchmark_60_40, name='60/40 Benchmark'),
]).set_index('Strategy')

print("BACKTEST PERFORMANCE SUMMARY")
print("=" * 70)
print(results_table.to_string())

BACKTEST PERFORMANCE SUMMARY
                 Ann. Return %  Total Return %  Volatility %  Sharpe Ratio  Sortino Ratio  Calmar Ratio  Max Drawdown %  Win Rate %
Strategy                                                                                                                           
Max Sharpe               24.56          273.55         12.66         1.545          2.510         1.379          -17.81        76.4
Min Variance              2.00           12.62          6.07        -0.494         -0.689         0.131          -15.33        59.7
Risk Parity               5.41           37.21          9.09         0.046          0.067         0.315          -17.21        59.7
Equal Weight              7.77           56.65         11.32         0.244          0.332         0.450          -17.27        66.7
60/40 Benchmark           5.62           38.82          7.99         0.077          0.160         0.327          -17.20        52.8


In [6]:
# Chart 1 — Cumulative Returns
# Growth of $100,000 invested in each strategy

initial_investment = 100000

fig = go.Figure()

strategies = {
    'Max Sharpe'    : bt_max_sharpe,
    'Min Variance'  : bt_min_variance,
    'Risk Parity'   : bt_risk_parity,
    'Equal Weight'  : bt_equal_weight,
    '60/40 Benchmark': benchmark_60_40
}

colors = ['#F44336','#00BCD4','#FF9800','#4CAF50','#9E9E9E']
styles = ['solid','solid','solid','solid','dash']

for i, (name, returns) in enumerate(strategies.items()):
    cum_value = (1 + returns).cumprod() * initial_investment

    fig.add_trace(go.Scatter(
        x    = cum_value.index,
        y    = cum_value.values,
        name = name,
        line = dict(
            width = 3 if name != '60/40 Benchmark' else 2,
            color = colors[i],
            dash  = styles[i]
        ),
        hovertemplate = f'<b>{name}</b><br>%{{x}}<br>Value: $%{{y:,.0f}}<extra></extra>'
    ))

fig.add_hline(y=initial_investment, line_dash='dot',
              line_color='white', opacity=0.3)

fig.update_layout(
    title      = dict(
        text=f'Growth of ${initial_investment:,} — All Strategies vs 60/40 Benchmark',
        font=dict(size=17)
    ),
    xaxis_title= 'Date',
    yaxis_title= 'Portfolio Value ($)',
    height     = 580,
    template   = 'plotly_dark',
    hovermode  = 'x unified',
    legend     = dict(orientation='h', y=-0.15),
    font       = dict(family='Arial', size=12)
)

fig.show()
print("Chart 1 complete — Cumulative returns")

Chart 1 complete — Cumulative returns


In [7]:
# Chart 2 — Drawdown comparison across all strategies

fig = go.Figure()

for i, (name, returns) in enumerate(strategies.items()):
    cum   = (1 + returns).cumprod()
    rm    = cum.cummax()
    dd    = (cum - rm) / rm * 100

    fig.add_trace(go.Scatter(
        x    = dd.index,
        y    = dd.values,
        name = name,
        line = dict(width=2, color=colors[i],
                    dash='dash' if name == '60/40 Benchmark' else 'solid'),
        fill = 'tozeroy',
        hovertemplate = f'<b>{name}</b><br>%{{x}}<br>Drawdown: %{{y:.1f}}%<extra></extra>'
    ))

fig.update_layout(
    title      = dict(
        text='Strategy Drawdowns Comparison',
        font=dict(size=18)
    ),
    xaxis_title= 'Date',
    yaxis_title= 'Drawdown (%)',
    height     = 520,
    template   = 'plotly_dark',
    hovermode  = 'x unified',
    legend     = dict(orientation='h', y=-0.2),
    font       = dict(family='Arial', size=12)
)

fig.show()
print("Chart 2 complete — Drawdown comparison")

Chart 2 complete — Drawdown comparison


In [8]:
# Chart 3 — Annual returns bar chart per strategy per year

annual_data = {}
for name, returns in strategies.items():
    annual = returns.resample('YE').apply(lambda x: (1+x).prod()-1) * 100
    annual.index = annual.index.year
    annual_data[name] = annual

annual_df = pd.DataFrame(annual_data)

fig = go.Figure()

bar_colors = ['#F44336','#00BCD4','#FF9800','#4CAF50','#9E9E9E']

for i, col in enumerate(annual_df.columns):
    fig.add_trace(go.Bar(
        name         = col,
        x            = annual_df.index.astype(str),
        y            = annual_df[col].round(1),
        marker_color = bar_colors[i],
        text         = annual_df[col].round(1).apply(lambda x: f'{x}%'),
        textposition = 'outside'
    ))

fig.add_hline(y=0, line_color='white', line_width=1, opacity=0.5)

fig.update_layout(
    title      = dict(
        text='Annual Returns by Strategy and Year (%)',
        font=dict(size=18)
    ),
    xaxis_title= 'Year',
    yaxis_title= 'Annual Return (%)',
    barmode    = 'group',
    height     = 520,
    template   = 'plotly_dark',
    legend     = dict(orientation='h', y=-0.2),
    font       = dict(family='Arial', size=12)
)

fig.show()
print("Chart 3 complete — Annual returns by year")

Chart 3 complete — Annual returns by year


In [9]:
# Chart 4 — Rolling 12 month Sharpe per strategy

fig = go.Figure()

for i, (name, returns) in enumerate(strategies.items()):
    rolling_sharpe = (
        returns.rolling(12).mean() * 12 - RISK_FREE_RATE
    ) / (returns.rolling(12).std() * np.sqrt(12))

    fig.add_trace(go.Scatter(
        x    = rolling_sharpe.index,
        y    = rolling_sharpe.values,
        name = name,
        line = dict(width=2, color=colors[i],
                    dash='dash' if name == '60/40 Benchmark' else 'solid')
    ))

fig.add_hline(y=0, line_color='white', line_width=1, opacity=0.4)
fig.add_hline(y=1, line_dash='dash', line_color='yellow',
              opacity=0.5, annotation_text='Sharpe = 1',
              annotation_position='right')

fig.update_layout(
    title      = dict(
        text='Rolling 12-Month Sharpe Ratio by Strategy',
        font=dict(size=18)
    ),
    xaxis_title= 'Date',
    yaxis_title= 'Rolling Sharpe Ratio',
    height     = 500,
    template   = 'plotly_dark',
    hovermode  = 'x unified',
    legend     = dict(orientation='h', y=-0.2),
    font       = dict(family='Arial', size=12)
)

fig.show()
print("Chart 4 complete — Rolling Sharpe by strategy")

Chart 4 complete — Rolling Sharpe by strategy


In [10]:
# Final tearsheet — clean summary table with all strategies

print("")
print("=" * 70)
print("   FULL PERFORMANCE TEARSHEET — MULTI-ASSET PORTFOLIO OPTIMIZER")
print("=" * 70)
print(f"   Backtest Period : {common_start.date()} to {bt_max_sharpe.index[-1].date()}")
print(f"   Rebalancing     : Monthly walk-forward")
print(f"   Universe        : 7 Asset Classes")
print(f"   Benchmark       : 60/40 (SPY/TLT)")
print("=" * 70)
print("")
print(results_table.to_string())
print("")
print("=" * 70)

# Highlight the winner
best_sharpe_strategy = results_table['Sharpe Ratio'].idxmax()
best_return_strategy = results_table['Ann. Return %'].idxmax()
least_dd_strategy    = results_table['Max Drawdown %'].idxmax()

print("")
print("KEY TAKEAWAYS")
print(f"  Best Risk-Adjusted Return : {best_sharpe_strategy}")
print(f"  Highest Absolute Return   : {best_return_strategy}")
print(f"  Smallest Max Drawdown     : {least_dd_strategy}")

results_table.to_csv(f'{save_path}backtest_results.csv')
print("")
print("Results saved to Google Drive")
print("Notebook 4 Complete - Next: Notebook 5 Final Report")


   FULL PERFORMANCE TEARSHEET — MULTI-ASSET PORTFOLIO OPTIMIZER
   Backtest Period : 2019-01-31 to 2024-12-31
   Rebalancing     : Monthly walk-forward
   Universe        : 7 Asset Classes
   Benchmark       : 60/40 (SPY/TLT)

                 Ann. Return %  Total Return %  Volatility %  Sharpe Ratio  Sortino Ratio  Calmar Ratio  Max Drawdown %  Win Rate %
Strategy                                                                                                                           
Max Sharpe               24.56          273.55         12.66         1.545          2.510         1.379          -17.81        76.4
Min Variance              2.00           12.62          6.07        -0.494         -0.689         0.131          -15.33        59.7
Risk Parity               5.41           37.21          9.09         0.046          0.067         0.315          -17.21        59.7
Equal Weight              7.77           56.65         11.32         0.244          0.332         0.450         